In [3]:
from flask import Flask, request, jsonify
from joblib import load
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from nltk.sentiment import SentimentIntensityAnalyzer

In [6]:
import nltk
nltk.download('vader_lexicon')


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...


True

In [7]:
app = Flask(__name__)

# Load the model and necessary components
model = load('models/random_forest_model.joblib')
vectorizer = load('models/vectorizer.joblib')  # For text-based features
sia = SentimentIntensityAnalyzer()  # Sentiment analysis

In [8]:
@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.json

        # Extract inputs
        description = data.get('description', '')
        synopsis = data.get('synopsis', '')
        crew_roles = data.get('crew_roles', [])
        casting_roles = data.get('casting_roles', [])
        production_dates = data.get('production_dates', [])
        production_locations = data.get('production_locations', [])

        # Preprocess inputs
        num_casting_roles = len(casting_roles)
        num_crew_roles = len(crew_roles)
        duration = (np.datetime64(production_dates[-1]) - np.datetime64(production_dates[0])).astype(int)
        collaboration_index = (num_casting_roles + num_crew_roles) / max(1, len(set(crew_roles + casting_roles)))
        collaboration_index_per_day = collaboration_index / max(1, duration)

        # Sentiment analysis
        description_sentiment = sia.polarity_scores(description)['compound']
        synopsis_sentiment = sia.polarity_scores(synopsis)['compound']

        # Vectorize text-based features
        text_features = vectorizer.transform([description + synopsis]).toarray()

        # Combine features into a single array
        features = np.array([collaboration_index_per_day, description_sentiment, synopsis_sentiment])
        features = np.hstack([features, text_features])

        # Predict
        prediction = model.predict([features])[0]
        return jsonify({'review_status': 'Approved' if prediction == 1 else 'Pending'})

    except Exception as e:
        return jsonify({'error': str(e)}), 500

if __name__ == '__main__':
    app.run(port=5001, debug=True)


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
 * Restarting with stat


SystemExit: 1

d:\Projects\3rd year group project\prod\prodstudio-ml\prod_env\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
